In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from pathlib import Path
import pandas as pd
import re
import parse_exercises_from_exam_file as parse

In [3]:
ROOT = Path("~/code/selector_examenes").expanduser()

Cargar los archivos json con las keywords.

In [4]:
def load_topic_keywords(json_file=ROOT / "JSON/quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

def load_topics_by_exercise_number(json_file=ROOT / "JSON/topics_based_on_exercise_number.json"):
    with open(json_file) as file:
        return json.load(file)

Normalizar el enunciado antes de buscar keywords: espacios y tildes.

In [5]:
def normalize_statement(statement: str) -> str:
    # convertir saltos de línea (y espacios repetidos) en un solo espacio
    statement = re.sub(r"\s+", " ", statement)

    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    return statement.translate(tabla)

Clasificación de tema de ejercicio, basado en puntuación de palabras claves que hacen *match*

In [6]:
def classify_exercise(statement: str, keywords = None) -> list:
    if keywords is None:
        keywords = load_topic_keywords()

    statement = normalize_statement(statement)

    topics = {}
    for topic in keywords.keys():
        topic_keywords = keywords[topic]["topic_kw"]
        pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
        keyword_matches = re.findall(pattern, statement)
        if len(keyword_matches) > 0:
            topics[topic] = len(keyword_matches)

    most_occurrences_topic = [k for k,v in topics.items() if v == max(topics.values())]

    return most_occurrences_topic

Devolver lista de posibles temas a los que puede pertenecer un ejercicio, basado en el número de ejercicio y el año del examen.

In [7]:
def get_candidate_topics(exercise_number: str, exam_year: int) -> list | None:

    possible_topics_dict = load_topics_by_exercise_number()

    # seleccionar modelo de examen en función del año
    if exam_year <= 2019:
        exam_model = "modelo_2010_2019"
        exercise_number_pattern = r"\d{1}"

    elif exam_year <= 2024:
        exam_model = "modelo_2020_2024"
        exercise_number_pattern = r"^([ABC])\d{1}"
    else:
        exam_model = "modelo_2025+"
        exercise_number_pattern = r"^\d{1}?[AB]|^5"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    topics_for_model_dict = possible_topics_dict[exam_model]

    # obtener el número del ejercicio
    key = re.search(exercise_number_pattern, exercise_number)

    # si se encuentra el número de ejercicio
    if key:

        key = key[0]
        # extraer la clave del diccionario del modelo de examen

        # 2010 - 2019 - key 1,2,3,...
        if exam_year <= 2019:  
            
            # las keys son: "1", "2,3,4", "5,6"
            if key in "2,3,4":
                key = "2,3,4"
            elif key in "5,6":
                    key = "5,6"
            else:
                key = "1"

        elif 2020 <= exam_year <= 2024:
            # extraer letra del ejercicio B1 --> B
            key = key[0]

        else:
            if key[0] != "4":
                key = key[0]

        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        formulation_pre_2020 = exam_year <= 2019 and key == "1"
        formulation_2020_2024 =  (2020 <= exam_year <= 2024) and key == "A"

        if formulation_pre_2020 or formulation_2020_2024:
            exercise_topics = ["Formulación"]

        elif exam_year >= 2025 and key == "4A":
            exercise_topics = ["Formulación"]

        else:
            # opciones de tema para el ejercicio suministrado
            exercise_topics = topics_for_model_dict[key]

        return exercise_topics

Devolver el tema al que pertenece el ejercicio, proporcionando la lista de posibles temas a los que puede pertenecer (salida de la función anterior).

In [8]:
def classify_among_candidates(statement: str, candidate_topics: list, keywords = None) -> list:
    if keywords is None:
        keywords = load_topic_keywords()

    statement = normalize_statement(statement)

    # un ejercicio puede ser de diferentes temas
    # elijo el tema por puntuación
    topic_scores = {}
    for topic in candidate_topics:
        topic_keywords = keywords[topic]["topic_kw"]
        pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
        kw_matches_count = len(re.findall(pattern, statement))
        if kw_matches_count > 0:
            topic_scores[topic] = kw_matches_count

    best_topics = [k for k,v in topic_scores.items() if v == max(topic_scores.values())]
    return best_topics

Extraer los ejercicios de un archivo de examen.

In [9]:
def exam_to_dataframe(exam_file: Path) -> pd.DataFrame:
    exercises_list = []

    # extraer el texto del examen
    exercises_text = parse.extract_exercises_from_pdf(exam_file)

    # clasificar el periodo del examen
    year = int(re.search(r"\d{4}", exam_file.name)[0])
    exam_period = parse.get_exam_period(year)

    # anadir los ejercicios a una lista de diccionarios para
    # luego crear el dataframe
    for ex in parse.split_exercises(exercises_text, exam_period):
        #topic = classify_exercise(statement=ex)
        exercises_list.append({"year" : year, "statement" : ex, "exam_file" : exam_file})
              
    return pd.DataFrame(exercises_list)

Extraer número de ejercicio a partir del texto del enunciado.

In [10]:
def get_exercise_number(row):
    if row.year <= 2019:
        exercise_number = row.statement[:2]
        statement = row.statement[3:].strip()

    elif row.year >= 2025:
        exercise_number = row.statement[:2]
        if exercise_number[0] == "5":
            exercise_number = "5"
        statement = row.statement[3:].strip()
    else:
        exercise_number = row.statement[:2]
        statement = row.statement[3:].strip()
    return exercise_number, statement

Extraer convocatoria del examen a partir del nombre del archivo.

In [28]:
def parse_convocatoria(exam_file):
    exam_file = Path(exam_file)
    if exam_file.name.startswith("química") or exam_file.name.startswith("quimica"):
        return exam_file.name.split(" - ")[-1].replace(".pdf", "").title()
    else:
        return exam_file.name

Clasificar ejercicios de un archivo

In [12]:
# file = ROOT / "examenes_quimica/2025_quimica/química - 2025 - julio.pdf"

In [13]:
# df = exam_to_dataframe(file)

## Clasificar todos los ejercicios de TODOS los exámenes

In [14]:
df = pd.DataFrame()
for file in (ROOT / "PDF/examenes_quimica").rglob("*.pdf"):
    df_file = exam_to_dataframe(file)
    df = pd.concat([df, df_file], axis=0)

df.to_csv("df.csv", index=False)

Guardo una copia en csv antes de generar el número de ejercicio y la convocatoria

In [23]:
df = pd.read_csv("df.csv")

In [26]:
df.head()

,year,statement,exam_file,exercise_number
0,2011,Formule o nombre los compuestos siguientes: a)...,/home/daniel/code/selector_examenes/PDF/examen...,1A
1,2011,Dadas las moléculas NH₃ y CCl₄:\na) Represente...,/home/daniel/code/selector_examenes/PDF/examen...,2A
2,2011,Se construye una pila conectando dos electrodo...,/home/daniel/code/selector_examenes/PDF/examen...,3A
3,2011,Razone qué ocurrirá con el pH cuando:\na) Se a...,/home/daniel/code/selector_examenes/PDF/examen...,4A
4,2011,El carbonato de magnesio reacciona con ácido c...,/home/daniel/code/selector_examenes/PDF/examen...,5A


Genero el número de ejercicio y la convocatoria

In [25]:
df["exercise_number"], df["statement"] = zip(*df.apply(get_exercise_number, axis=1))

In [29]:
df["convocatoria"] = df.exam_file.apply(parse_convocatoria)

## Clasifico los ejercicios

In [30]:
df.head()

,year,statement,exam_file,exercise_number,convocatoria
0,2011,Formule o nombre los compuestos siguientes: a)...,/home/daniel/code/selector_examenes/PDF/examen...,1A,2011_quimica_sel_2011_quimica - 222-Examen_And...
1,2011,Dadas las moléculas NH₃ y CCl₄:\na) Represente...,/home/daniel/code/selector_examenes/PDF/examen...,2A,2011_quimica_sel_2011_quimica - 222-Examen_And...
2,2011,Se construye una pila conectando dos electrodo...,/home/daniel/code/selector_examenes/PDF/examen...,3A,2011_quimica_sel_2011_quimica - 222-Examen_And...
3,2011,Razone qué ocurrirá con el pH cuando:\na) Se a...,/home/daniel/code/selector_examenes/PDF/examen...,4A,2011_quimica_sel_2011_quimica - 222-Examen_And...
4,2011,El carbonato de magnesio reacciona con ácido c...,/home/daniel/code/selector_examenes/PDF/examen...,5A,2011_quimica_sel_2011_quimica - 222-Examen_And...


In [31]:
exercise_topics = []
for idx, row in df.iterrows():
    possible_topics = get_candidate_topics(row.exercise_number, row.year)
    
    if possible_topics :
        # formulación
        if possible_topics[0] == "Formulación":
            exercise_topics.append(["Formulación"])
        else:
            topics = classify_among_candidates(row.statement, possible_topics)
            exercise_topics.append(topics)

df["topic"] = exercise_topics
df = df.explode("topic")


### Inspección de los resultados de la clasificación

In [32]:
df.shape

(1274, 6)

In [33]:
df.isna().sum()

year               0
statement          0
exam_file          0
exercise_number    0
convocatoria       0
topic              1
dtype: int64

In [34]:
df[df.topic.isna()]

,year,statement,exam_file,exercise_number,convocatoria,topic
664,2012,e mezclan 2 litros de cloro gas medidos a 97 º...,/home/daniel/code/selector_examenes/PDF/examen...,6B,2012_quimica_sel_2012_quimica - 2 QUÍMICA - mo...,NaN


In [35]:
df.to_csv(ROOT / "CSV/all_exercises_classified.csv", index=False, encoding="utf-8")

### Inspección de los ejercicios NO clasificados

In [36]:
df_no_topic = df[df.topic.isna()]

In [37]:
df_no_topic

,year,statement,exam_file,exercise_number,convocatoria,topic
664,2012,e mezclan 2 litros de cloro gas medidos a 97 º...,/home/daniel/code/selector_examenes/PDF/examen...,6B,2012_quimica_sel_2012_quimica - 2 QUÍMICA - mo...,NaN


In [23]:
# with open("whole_classification.txt", "w") as file:
#     for _, row in df_no_topic.iterrows():
#         item = f"{row.year} - {row.exam} - {row.exercise_number}\n\n"
#         item += row.statement
#         item += "\n\n"
#         file.write(item)

Reintento de clasificar los que quedaron sin clasificar:

In [24]:
# exercise_topics = []
# for idx, row in df_no_topic.iterrows():
#     possible_topics = get_candidate_topics(row.exercise_number, row.year)
    
#     if possible_topics :
#         # formulación
#         if possible_topics[0] == "Formulación":
#             exercise_topics.append(["Formulación"])
#         else:
#             topics = classify_among_candidates(row.statement, possible_topics)
#             exercise_topics.append(topics)

# df_no_topic["topic"] = exercise_topics
# df_no_topic = df_no_topic.explode("topic")

In [25]:
# df_no_topic[["exam", "exercise_number", "statement"]]